# 2.1

1.返回\x00

In [7]:
chr(0)


'\x00'

2.多了一条杠

In [10]:
chr(0).__repr__()

"'\\x00'"

3. chr(0)在print中不显示

In [12]:
print(chr(0))

 


In [13]:
"this is a test" + chr(0) + "string"

'this is a test\x00string'

In [14]:

print("this is a test" + chr(0) + "string")

this is a test string


chr 和 ord 都是unicode字符级别的处理，稀疏，词表巨大

In [17]:
ord('我')

25105

# 2.2
1. utf对于ASCILL范围内的字符保持原样(0-255)。

utf-8是编码后最短的，其他两个会有许多0


In [25]:
test_string=" hello!我喜欢你！"
test_utf8=list(test_string.encode("utf-8"))
test_utf16=list(test_string.encode("utf-16"))
test_utf32=list(test_string.encode("utf-32"))
print(len(test_utf8))
print(len(test_utf16))
print(len(test_utf32))

22
26
52


2. 这个是一个字节一个字节解码的，当bytestring中有超出ASCIL码范围的字节时，转化就会失败。这里utf-8只接受0-127范围内的(因为它是一个字节的范围)，中文超出时就有错误

In [33]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])
decode_utf8_bytes_to_str_wrong("hello 我".encode("utf-8"))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe6 in position 0: unexpected end of data

3.选择128范围以外的就不行

In [44]:
print(b'\x80\x80'.decode())

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x80 in position 0: invalid start byte

# 2.4

正确版本是：

用 pre-tokenizer 划分 pre-token。

Counter(pre_token)，得到每个 pre-token 的出现次数。

把每个 pre-token 编码成 UTF-8 bytes，再拆成 tuple[bytes, ...]。

在每个 pre-token 的当前 token 序列内部统计相邻 pair，乘以这个 pre-token 的次数，加到全局 pair count。

选出全局次数最高的 pair，把这个 pair 记录为一条 merge rule，并把两个 bytes/token 拼接成一个新的 token。

在所有 pre-token 序列中，把出现的这个 pair 合并。

回到第 4 步，重新统计 pair。


In [117]:
def merge(word_tuple_list,pair,idx):
    
    new_word_tuple_list=[]
    for word_tuple in word_tuple_list:
        tmp=[]
        i=0
        while i < len(word_tuple):
            if (i<len(word_tuple)-1) and word_tuple[i]==pair[0] and word_tuple[i+1]==pair[1]:
                tmp.append(idx)
                i+=2
                continue
            else:
                tmp.append(word_tuple[i])
            i+=1
        new_word_tuple_list.append(tuple(tmp))
    return new_word_tuple_list
    

In [125]:
import regex,re
from collections import Counter

text="low Terry widest low widest widest newest newest low low low lower lower newest newest newest newest"
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

tokens=[match.group() for match in regex.finditer(PAT,text)]

print(tokens)
#先统计每个词的出现次数
word_count=Counter(tokens)


word_tuple_list=[]
for word in word_count.keys():
    word_tuple=tuple(b for b in word.encode("utf-8"))
    word_tuple_list.append(word_tuple)
print(word_tuple_list)
count=list(word_count.values())
idx=256
for k in range(6):

    All_count=Counter()
    for i,tuple_word in enumerate(word_tuple_list):
        dict=Counter(zip(tuple_word,tuple_word[1:]))
        All_count.update({pair:count[i]*c for pair,c in dict.items()})

    most_common_pair=All_count.most_common(1)[0][0]
    word_tuple_list=merge(word_tuple_list,most_common_pair,idx)
    idx+=1


print(word_tuple_list)

['low', ' Terry', ' widest', ' low', ' widest', ' widest', ' newest', ' newest', ' low', ' low', ' low', ' lower', ' lower', ' newest', ' newest', ' newest', ' newest']
[(108, 111, 119), (32, 84, 101, 114, 114, 121), (32, 119, 105, 100, 101, 115, 116), (32, 108, 111, 119), (32, 110, 101, 119, 101, 115, 116), (32, 108, 111, 119, 101, 114)]
[(259,), (32, 84, 101, 114, 114, 121), (32, 119, 105, 100, 257), (260,), (261, 101, 119, 257), (260, 101, 114)]


# 2.5
最长的是enthusiastically

In [3]:
import pickle
with open("vocab.pkl", "rb") as f:
    vocab=pickle.load(f)
    lon=max(vocab.items(),
        key=lambda x:len(x[1]))
    print(lon)

(9210, b' enthusiastically')


In [4]:
with open("vocab_32000.pkl", "rb") as f:
    vocab3=pickle.load(f)
    lon=max(vocab.items(),
        key=lambda x:len(x[1]))
    print(lon)

with open("merges_32000.pkl", "rb") as f:
    merges3=pickle.load(f)
  

(9210, b' enthusiastically')


In [6]:
print(len(vocab3))
print(len(merges3))

27883
27626


# 2.6

In [15]:
with open("D:/HuaweiMoveData/Users/huawei/Desktop/AI知识库/CS336/assignment1-basics/data/TinyStoriesV2-GPT4-train.txt","rb") as f:
    text=f.read()
with open("D:/HuaweiMoveData/Users/huawei/Desktop/AI知识库/CS336/assignment1-basics/cs336_basics/tiny_story_ids.pkl","rb") as f:
    ids=pickle.load(f)

In [18]:
print(f"压缩率:{len(text)/(len(ids))}")

压缩率:4.150244577727777


# 3.5